**Claims Textract POC**

Run each cell top to bottom. Update `aws_profile`, `bucket`, or `key` if needed.

**Setup**

In [ ]:
import json
import time
from pathlib import Path

import boto3

aws_region = "us-east-2"
aws_profile = "aws-profile-of-yours"

bucket = "REPLACE_WITH_YOUR_BUCKET_NAME"
key = "raw/sample-claim.pdf"

session = boto3.Session(
    profile_name=aws_profile,
    region_name=aws_region,
)

s3 = session.client("s3")
textract = session.client("textract")

print(f"Input: s3://{bucket}/{key}")

**Upload sample PDF**

In [ ]:
sample_pdf_path = Path("sample-claim.pdf").resolve()

if not sample_pdf_path.exists():
    raise FileNotFoundError(f"Could not find sample PDF: {sample_pdf_path}")

s3.upload_file(
    Filename=str(sample_pdf_path),
    Bucket=bucket,
    Key=key,
)

print(f"Uploaded {sample_pdf_path} to s3://{bucket}/{key}")

**Start Textract**

In [ ]:
response = textract.start_document_analysis(
    DocumentLocation={
        "S3Object": {
            "Bucket": bucket,
            "Name": key,
        }
    },
    FeatureTypes=["FORMS", "TABLES"],
)

job_id = response["JobId"]
job_id

**Poll status**

In [ ]:
while True:
    response = textract.get_document_analysis(JobId=job_id)
    status = response["JobStatus"]
    print(f"Textract status: {status}")

    if status in ["SUCCEEDED", "FAILED"]:
        break

    time.sleep(5)

if status == "FAILED":
    raise RuntimeError("Textract job failed")

**Fetch blocks**

In [ ]:
blocks = []
next_token = None

while True:
    if next_token:
        response = textract.get_document_analysis(JobId=job_id, NextToken=next_token)
    else:
        response = textract.get_document_analysis(JobId=job_id)

    blocks.extend(response["Blocks"])
    next_token = response.get("NextToken")

    if not next_token:
        break

print(f"Total blocks received: {len(blocks)}")

**Lines**

In [ ]:
lines = [
    {
        "text": block.get("Text", ""),
        "confidence": block.get("Confidence", 0),
    }
    for block in blocks
    if block["BlockType"] == "LINE"
]

print(f"Total lines: {len(lines)}")
lines[:20]

**Parse helpers**

In [ ]:
def get_text(block, block_map):
    text = []

    for relationship in block.get("Relationships", []):
        if relationship["Type"] != "CHILD":
            continue

        for child_id in relationship["Ids"]:
            child = block_map.get(child_id)

            if not child:
                continue

            if child["BlockType"] == "WORD":
                text.append(child.get("Text", ""))
            elif child["BlockType"] == "SELECTION_ELEMENT":
                if child.get("SelectionStatus") == "SELECTED":
                    text.append("SELECTED")

    return " ".join(text).strip()


def extract_key_values(blocks):
    block_map = {block["Id"]: block for block in blocks}
    key_map = {}
    value_map = {}

    for block in blocks:
        if block["BlockType"] != "KEY_VALUE_SET":
            continue

        entity_types = block.get("EntityTypes", [])

        if "KEY" in entity_types:
            key_map[block["Id"]] = block
        elif "VALUE" in entity_types:
            value_map[block["Id"]] = block

    key_values = {}

    for key_block in key_map.values():
        key_text = get_text(key_block, block_map)
        value_text = ""

        for relationship in key_block.get("Relationships", []):
            if relationship["Type"] != "VALUE":
                continue

            for value_id in relationship["Ids"]:
                value_block = value_map.get(value_id)

                if value_block:
                    value_text = get_text(value_block, block_map)

        if key_text:
            key_values[key_text] = value_text

    return key_values


def extract_tables(blocks):
    block_map = {block["Id"]: block for block in blocks}
    tables = []

    for block in blocks:
        if block["BlockType"] != "TABLE":
            continue

        table = {}

        for relationship in block.get("Relationships", []):
            if relationship["Type"] != "CHILD":
                continue

            for child_id in relationship["Ids"]:
                cell = block_map.get(child_id)

                if not cell or cell["BlockType"] != "CELL":
                    continue

                row_index = cell["RowIndex"]
                col_index = cell["ColumnIndex"]
                table.setdefault(row_index, {})
                table[row_index][col_index] = get_text(cell, block_map)

        rows = []
        for row_index in sorted(table.keys()):
            row = []
            for col_index in sorted(table[row_index].keys()):
                row.append(table[row_index][col_index])
            rows.append(row)

        tables.append(rows)

    return tables

**Key-values**

In [ ]:
key_values = extract_key_values(blocks)

print(f"Total key-value pairs: {len(key_values)}")
dict(list(key_values.items())[:20])

**Tables**

In [ ]:
tables = extract_tables(blocks)

print(f"Total tables: {len(tables)}")
tables[:2]

**Claim JSON**

In [ ]:
def find_value(key_values, possible_keys):
    for search_key in possible_keys:
        for actual_key, value in key_values.items():
            if search_key.lower() in actual_key.lower():
                return value

    return None


def extract_service_lines_from_tables(tables):
    service_lines = []

    for table in tables:
        if not table:
            continue

        header = [h.lower() for h in table[0]]
        has_date_column = any("date" in h or "dos" in h for h in header)
        has_code_column = any(
            "code" in h or "cpt" in h or "procedure" in h or "hcpcs" in h
            for h in header
        )

        if not (has_date_column and has_code_column):
            continue

        for row in table[1:]:
            line = {}

            for index, header_name in enumerate(header):
                value = row[index] if index < len(row) else None

                if not value:
                    continue

                if "date" in header_name or "dos" in header_name:
                    line["dateOfService"] = value
                elif (
                    "cpt" in header_name
                    or "procedure" in header_name
                    or "hcpcs" in header_name
                    or "code" in header_name
                ):
                    line["procedureCode"] = value
                elif "unit" in header_name:
                    line["units"] = value
                elif "billed" in header_name or "charge" in header_name:
                    line["billedAmount"] = value
                elif "allowed" in header_name:
                    line["allowedAmount"] = value
                elif "paid" in header_name:
                    line["paidAmount"] = value
                elif "patient" in header_name and "responsibility" in header_name:
                    line["patientResponsibility"] = value

            if line:
                service_lines.append(line)

    return service_lines


claim_json = {
    "document": {
        "sourceBucket": bucket,
        "sourceKey": key,
    },
    "claim": {
        "claimId": find_value(key_values, ["claim number", "claim id", "claim no", "claim"]),
        "memberId": find_value(key_values, ["member id", "member number", "subscriber id", "patient account"]),
        "patientName": find_value(key_values, ["patient name", "member name", "insured name"]),
        "providerName": find_value(key_values, ["provider name", "billing provider", "rendering provider"]),
        "dateOfService": find_value(key_values, ["date of service", "service date", "dos"]),
        "totalBilledAmount": find_value(key_values, ["billed amount", "total charge", "charges", "amount billed"]),
        "totalPaidAmount": find_value(key_values, ["paid amount", "payment amount", "amount paid"]),
        "serviceLines": extract_service_lines_from_tables(tables),
    },
    "raw": {
        "keyValues": key_values,
        "tables": tables,
        "lines": lines,
    },
}

print(json.dumps(claim_json, indent=2))

**Save JSON**

In [ ]:
def build_output_key(input_key):
    if input_key.startswith("raw/"):
        output_key = input_key.replace("raw/", "output/", 1)
    else:
        output_key = f"output/{input_key}"

    if output_key.lower().endswith(".pdf"):
        output_key = output_key[:-4] + ".json"
    else:
        output_key = output_key + ".json"

    return output_key


output_key = build_output_key(key)

s3.put_object(
    Bucket=bucket,
    Key=output_key,
    Body=json.dumps(claim_json, indent=2),
    ContentType="application/json",
)

print(f"Saved JSON to s3://{bucket}/{output_key}")